Simulates a single frame with a red laser line at a given height. and   Detects red laser line using HSV thresholding.


In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def extract_frames(video_path):
    """
    Extracts frames from the input video.
    """
    cap = cv2.VideoCapture(video_path)
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames

def detect_laser_line(img):
    """
    Detects red laser line using HSV thresholding.
    """
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower_red1 = np.array([0, 100, 100])
    upper_red1 = np.array([10, 255, 255])
    lower_red2 = np.array([160, 100, 100])
    upper_red2 = np.array([179, 255, 255])
    mask = cv2.bitwise_or(cv2.inRange(hsv, lower_red1, upper_red1),
                          cv2.inRange(hsv, lower_red2, upper_red2))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))
    coords = cv2.findNonZero(mask)
    return coords

def measure_depth_from_video(frames, cm_to_px=50):
    """
    Measures depth from laser line position across video frames.
    """
    measured_px = []
    for frame in frames:
        coords = detect_laser_line(frame)
        if coords is not None:
            avg_y = np.mean(coords[:, 0, 1])
            measured_px.append(avg_y)
        else:
            measured_px.append(np.nan)
    return np.array(measured_px)

def save_plot(measured_cm, filename="results/tread_depth_plot.png"):
    plt.figure(figsize=(10, 5))
    plt.plot(measured_cm, label="Measured Tread Profile (cm)", linestyle='--', color='red')
    plt.title("Tread Depth Profile from Laser Line")
    plt.xlabel("Frame Index")
    plt.ylabel("Height (cm)")
    plt.legend()
    plt.grid(True)
    os.makedirs("results", exist_ok=True)
    plt.savefig(filename)
    plt.close()

def export_csv(measured_cm, filename="results/tread_profile.csv"):
    np.savetxt(filename, measured_cm, delimiter=",", header="Measured Depth (cm)", comments='')

def main():
    print("🔴 Processing Tire Pan 2.mp4 for tread depth estimation...")
    video_path = "Tire Pan 2.mp4"
    cm_to_px = 50.0  # Adjust based on calibration

    frames = extract_frames(video_path)
    measured_px = measure_depth_from_video(frames, cm_to_px)
    measured_cm = measured_px / cm_to_px

    max_depth = np.nanmax(measured_cm)
    min_depth = np.nanmin(measured_cm)
    tread_depth = max_depth - min_depth

    print("\n📊 Results")
    print(f"Max height: {max_depth:.2f} cm")
    print(f"Min height: {min_depth:.2f} cm")
    print(f"Estimated tread depth: {tread_depth:.2f} cm")

    save_plot(measured_cm)
    export_csv(measured_cm)

if __name__ == "__main__":
    main()


🔴 Processing Tire Pan 2.mp4 for tread depth estimation...

📊 Results
Max height: 18.42 cm
Min height: 12.56 cm
Estimated tread depth: 5.87 cm


# Algorithm validation

In [5]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def simulate_laser_frame(tread_height_cm, cm_to_px=50, width=640, height=480):
    """
    Simulates a single frame with a red laser line at a given height.
    """
    img = np.zeros((height, width, 3), dtype=np.uint8)
    y_pos = int(tread_height_cm * cm_to_px)
    cv2.line(img, (0, y_pos), (width - 1, y_pos), (0, 0, 255), 5)
    return img

def detect_laser_line(img):
    """
    Detects red laser line using HSV thresholding.
    """
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower_red1 = np.array([0, 100, 100])
    upper_red1 = np.array([10, 255, 255])
    lower_red2 = np.array([160, 100, 100])
    upper_red2 = np.array([179, 255, 255])
    mask = cv2.bitwise_or(cv2.inRange(hsv, lower_red1, upper_red1),
                          cv2.inRange(hsv, lower_red2, upper_red2))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))
    coords = cv2.findNonZero(mask)
    return coords

def measure_depth(profile_cm, cm_to_px=50):
    """
    Measures depth from simulated laser line positions.
    """
    measured_px = []
    for h in profile_cm:
        frame = simulate_laser_frame(h, cm_to_px)
        coords = detect_laser_line(frame)
        if coords is not None:
            avg_y = np.mean(coords[:, 0, 1])
            measured_px.append(avg_y)
        else:
            measured_px.append(np.nan)
    return np.array(measured_px)

def save_plot(actual_cm, measured_cm, filename="results/simulated_plot.png"):
    plt.figure(figsize=(10, 5))
    plt.plot(actual_cm, label="Actual Tread Profile (cm)")
    plt.plot(measured_cm, '--', label="Measured Profile (cm)")
    plt.title("Simulated Tread Depth via Laser Line")
    plt.xlabel("Frame Index")
    plt.ylabel("Height (cm)")
    plt.legend()
    plt.grid(True)
    os.makedirs("results", exist_ok=True)
    plt.savefig(filename)
    plt.close()

def export_csv(measured_cm, filename="results/measured_profile.csv"):
    np.savetxt(filename, measured_cm, delimiter=",", header="Measured Depth (cm)", comments='')

def main():
    print("🔴 Starting structured light simulation...")
    num_frames = 100
    x = np.arange(num_frames)
    profile_cm = 0.8 * np.sin(x / 10) + 1.2  # Simulated tread profile
    cm_to_px = 50.0

    measured_px = measure_depth(profile_cm, cm_to_px)
    measured_cm = measured_px / cm_to_px

    max_depth = np.nanmax(measured_cm)
    min_depth = np.nanmin(measured_cm)
    tread_depth = max_depth - min_depth

    print("\n📊 Simulation Results")
    print(f"Max height: {max_depth:.2f} cm")
    print(f"Min height: {min_depth:.2f} cm")
    print(f"Estimated tread depth: {tread_depth:.2f} cm")

    save_plot(profile_cm, measured_cm)
    export_csv(measured_cm)

if __name__ == "__main__":
    main()


🔴 Starting structured light simulation...

📊 Simulation Results
Max height: 1.98 cm
Min height: 0.40 cm
Estimated tread depth: 1.58 cm
